<a href="https://colab.research.google.com/github/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/01-model-apis/04-context-and-caching.zh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 上下文与缓存

**目标：** 给上下文窗口做预算、估算成本，并理解缓存与批量定价（这些概念你在任何厂商那里都会遇到）。

本项目是 [AI-EDF-NOTEBOOKS-CN](https://github.com/shangbingo888/AI-EDF-NOTEBOOKS-CN) 的一部分。


## 准备工作

每个 notebook 都是自包含的，所以接下来两个单元格会从零把它搭起来：

1. **安装依赖。** `aien` 包（本仓库）带着共享的 setup 辅助函数，并拉入 `groq` 客户端；本 notebook 还会用到 `tiktoken`。
2. **加载你的 API key。** 到 [console.groq.com](https://console.groq.com/) 领免费 key（无需信用卡）。在 Colab 里，通过左侧边栏的**钥匙图标** → **Add new secret** 添加，名称严格写成 `GROQ_API_KEY`，粘贴其值，并打开 **Notebook access** 开关。在本地运行？把 `GROQ_API_KEY` 设为环境变量。

（完整讲解与选模型指引见 [00-setup/00-environment.ipynb](https://colab.research.google.com/github/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/00-setup/00-environment.zh.ipynb)。）

In [ ]:
%pip install -q "git+https://github.com/shangbingo888/AI-EDF-NOTEBOOKS-CN.git@main" tiktoken

In [ ]:
from aien import setup

# 读取 GROQ_API_KEY（Colab Secrets 或本地环境变量），返回一个就绪的
# Groq 客户端。传 model=... 可覆盖默认值；如果之后某次调用报 404，
# 就列出可用模型——见 00-setup/00-environment.ipynb。
client, MODEL = setup()

## 心智模型：每个请求一份 token 预算

每个请求都从同一份预算里花钱：上下文窗口（`gpt-oss-120b` 上是 128k token，不过你很少想把它填满；成本与延迟都随你发送的内容而增长）。把每个请求想象成一本有四个条目的账：

| 条目 | 典型占比 | 说明 |
|---|---|---|
| System prompt | 固定，偏小 | 指令、人格设定、工具定义 |
| 检索来的上下文 | 最大的变量 | RAG 分块、文档、代码 |
| 对话历史 | 每回合都在增长 | 最终会把一切都吃掉的那个东西 |
| 输出余量 | 一定要预留！ | `max_tokens` 也算在窗口里 |

真正的失败模式不是撞上硬上限，而是成本悄悄爬升：一个 agent 重发 5 万 token 的历史，只为回答一个 10 token 的问题。做预算就意味着逐条测量。

Groq 的 OpenAI 兼容 API 没有专门的 `count_tokens` 端点。两种实用做法：

1. **事后精确：** 每个响应都带 `usage.prompt_tokens`，即你发送内容的真实计数。免费，但你要在调用*之后*才知道。
2. **事前估算：** 用 `tiktoken`（OpenAI 的分词器）作为*估计*。这里的模型分词方式不同，所以把它当成粗略值（通常误差在 ~10-20% 以内），用于飞行前的预算检查，真实数字仍以 `usage` 为准。

In [ ]:
import tiktoken

# tiktoken 是 OpenAI 的 cl100k 分词器，不会和你正在调用的模型完全一致
# ——所以这只是估算，用于飞行前的预算检查足够了。要拿真实
# 数字，就从响应的 usage.prompt_tokens 里读。
_enc = tiktoken.get_encoding('cl100k_base')


def count(messages, system=None):
    """估算一份消息列表（外加可选的 system 字符串）的 prompt token 数。"""
    text = ''
    if system:
        text += system + '\n'
    for m in messages:
        content = m['content'] if isinstance(m['content'], str) else str(m['content'])
        text += content + '\n'
    return len(_enc.encode(text))


SYSTEM = 'You are a support assistant for AcmeDB. Answer only from the provided runbook. Cite section numbers.'

print('system + 1 short question (approx):',
      count([{'role': 'user', 'content': 'How do I rotate credentials?'}], system=SYSTEM))

# 同样的内容，两种表述——token 取决于文本的字节数，而不是「想法」
terse = 'rotate creds acmedb how'
prose = ('Hello! I was wondering if you could kindly walk me through the process of '
         'rotating the credentials for our AcmeDB instance when you get a chance?')
print('terse phrasing: ', count([{'role': 'user', 'content': terse}]))
print('prose phrasing: ', count([{'role': 'user', 'content': prose}]))

# 现在从 API 拿精确计数，用作对比：
r = client.chat.completions.create(
    model=MODEL, max_tokens=5,
    messages=[{'role': 'system', 'content': SYSTEM},
              {'role': 'user', 'content': 'How do I rotate credentials?'}],
)
print('exact prompt_tokens from usage:', r.usage.prompt_tokens)

In [ ]:
# 一个可以留着的预算检查器：这个请求装得下预算吗？
def check_budget(system, context, history, max_output,
                 budget={'system': 2_000, 'context': 8_000, 'history': 4_000}):
    # 全部是估算（tiktoken），不发 API 调用。真实数字以 usage.prompt_tokens 为准。
    lines = {
        'system':  count([], system=system),
        'context': count([{'role': 'user', 'content': context}]),
        'history': count(history) if history else 0,
    }
    for name, used in lines.items():
        cap = budget[name]
        flag = 'OK ' if used <= cap else 'OVER'
        print(f'{flag} {name:8s} {used:>7,} / {cap:,}')
    total = sum(lines.values()) + max_output
    print(f'    total incl. {max_output:,} output headroom: {total:,} tokens')
    return lines


_ = check_budget(
    system=SYSTEM,
    context='Section 4.2: To rotate credentials, run acmedb rotate --all ...' * 40,
    history=[{'role': 'user', 'content': 'hi'}, {'role': 'assistant', 'content': 'Hello! How can I help?'}],
    max_output=1_000,
)


## prompt 缓存（一个需要了解的概念）

如果每个请求都重发同样一大段 system prompt 或文档，你就是在为重新处理模型几秒前刚看过的字节付全价。**prompt 缓存**解决了这个问题：厂商把已处理的前缀缓存起来，复用时只收很小一部分费用。

Groq 目前没有开放 prompt 缓存的 API，但你在其他地方到处都会遇到（Anthropic 的 `cache_control`、OpenAI 的自动前缀缓存、Google 的 context caching），所以值得把其中的经济账搞清楚：

- **它是对精确字节做前缀匹配。** 缓存点*之前*的任何字节变化都会让后面全部失效。把易变内容（时间戳、用户名）放在稳定前缀*之后*，绝不要放在里面。
- **典型经济账：** 缓存*写入*约为输入费率的 1.25 倍；缓存*读取*约为 0.1 倍。第二次请求就回本，此后重复的前缀享受约 90% 的折扣。
- **TTL 很短**（通常约 5 分钟，每次命中都会刷新）。

下面的单元格在 Groq 上演示缓存所解决的*问题*：同一个 8k token 的 runbook 前缀发两次，盯着 `prompt_tokens`。这些 token 你每一次调用都要付一次钱。在有缓存的厂商那里，第二次调用的前缀会便宜约 10 倍。


In [ ]:
# 构造一个确定性的、约 8k token 的 'runbook'。
sections = []
for i in range(1, 121):
    sections.append(
        f'Section {i}: Procedure AC-{i:03d}. To perform maintenance task {i}, first '
        f'verify replica lag is under {i * 5} ms, then drain node group {i % 7}, apply '
        f'the configuration bundle, and re-enable traffic. Rollback: restore snapshot '
        f'tagged ac-{i:03d}-pre and replay the WAL from the checkpoint marker.'
    )
RUNBOOK = '\n'.join(sections)

print('runbook tokens (approx):',
      count([{'role': 'user', 'content': RUNBOOK}]))


In [ ]:
def ask_runbook(question):
    return client.chat.completions.create(
        model=MODEL,
        max_tokens=150,
        messages=[
            {'role': 'system',
             'content': 'Answer from this runbook only, citing section numbers.\n\n' + RUNBOOK},
            {'role': 'user', 'content': question},
        ],
    )


# 2 次 API 调用，同一个大前缀。盯住 prompt_tokens：两次几乎一样——
# 你每次调用都要为整份 runbook 重新付钱。这正是 prompt 缓存要去掉的东西。
for question in ['What is the rollback for procedure AC-042?',
                 'Which node group does task 10 drain?']:
    r = ask_runbook(question)
    u = r.usage
    print(f'Q: {question}')
    print(f'   prompt_tokens={u.prompt_tokens}  completion_tokens={u.completion_tokens}')
    print(f'   A: {r.choices[0].message.content[:110]}...')
    print()


跑一下，注意两次调用的 `prompt_tokens` 基本一样：你为整份 runbook 付了两次钱。在有缓存的厂商那里，第二次调用的大前缀只会按约 10% 计费。当你*确实*有缓存可用时，下面这些会悄悄把你的命中率清零（请求仍然能跑通，只是你付全价）：

- 把时间戳、请求 ID 或用户名插进稳定前缀里，导致前缀每次请求都不同。把易变内容放在被缓存段落*之后*。
- 增删或重排 **tools**，它们渲染在 system prompt 之前。要确定性地序列化它们。
- 对话中途切换**模型**，因为缓存是按模型分的。
- 非确定性的序列化（`json.dumps` 不带 `sort_keys`、遍历一个 set）。

## Batch API：用耐心换半价（另一个概念）

实时调用（上面讲的一切）按全价计费。大多数主流厂商还提供 **Batch API**：提交很多请求，异步拿结果（几分钟到 24 小时），只需标准价格的约 **50%**。Groq 目前没有提供，但这套模式对应着一类很具体的 FDE/AI 工程师工作：**批量 eval**（给 500 份记录重新打分）、**回填**（给所有历史工单分类）、每夜的富化、回归扫描。判断信号总是一样的：没人在等这个输出，而且量很大。如果有人在盯着转圈图标，那 batch 就不合适；如果它是个定时任务，那就是白捡的钱。

既然我们在 Groq 上跑不了 batch，我们就来做*算术*，算清每种杠杆何时划算——这才是在设计评审里真正重要的技能。


### 在提供这些能力的厂商那里，它们实际长什么样

Groq 跑不了缓存和 batch，但你在 OpenAI/Anthropic 上会一直用到两者，所以这里给出*真实的* API 形态作参考。**别运行这些**（它们需要那些 SDK 和付费 key）；读一读，这样你第一次需要它们时不至于陌生。

**Anthropic prompt caching ——** 在内容块上用 `cache_control` 标记一个缓存断点；它*之前并包含它*的所有内容都会被缓存（顺序是 tools → system → messages）。`usage` 字段会告诉你有没有命中。

```python
# pip install anthropic ——仅作参考，这里不运行
import anthropic
client = anthropic.Anthropic()

resp = client.messages.create(
    model="claude-sonnet-5",
    max_tokens=150,
    system=[
        {"type": "text", "text": "Answer from this runbook only, cite section numbers."},
        {"type": "text",
         "text": RUNBOOK,                              # 那个大而稳定的前缀
         "cache_control": {"type": "ephemeral"}},      # <-- 缓存到这里为止的全部内容
    ],
    messages=[{"role": "user", "content": "What is the rollback for AC-042?"}],
)
u = resp.usage
# 第一次调用：cache_creation_input_tokens = <runbook>,  cache_read_input_tokens = 0   （约 1.25 倍的写入）
# 之后调用：  cache_creation_input_tokens = 0,          cache_read_input_tokens = <runbook>  （约 0.1 倍的读取）
print(u.cache_creation_input_tokens, u.cache_read_input_tokens, u.input_tokens)
```

前面那些单元格教你的纪律，正是让它产生回报的关键：把易变字节（用户问题）放在被标记的块*之后*，否则每个请求都是一次缓存未命中。（OpenAI 的前缀缓存是*自动的*，没有标记；你只要保持 prompt 前缀稳定，并读取 usage 里的 `cached_tokens` 字段即可。）

**OpenAI Batch API ——** 把你的请求写进一个 `.jsonl`（每行一个，每个都带 `custom_id`），上传它，创建 batch，轮询，然后下载结果。半价、异步，适合没人在等的工作。

```python
# pip install openai  ——仅作参考，这里不运行
from openai import OpenAI
client = OpenAI()

# 1. 构造一个 .jsonl：每行一个请求，每个都带一个 custom_id，稍后
#    用它把结果映射回来（输出顺序不保证与输入一致）。
#    每行形状：{"custom_id": "...", "method": "POST",
#                 "url": "/v1/chat/completions",
#                 "body": {"model": "...", "messages": [...]}}

# 2. 上传它并启动 batch。
f = client.files.create(file=open("requests.jsonl", "rb"), purpose="batch")
batch = client.batches.create(
    input_file_id=f.id,
    endpoint="/v1/chat/completions",
    completion_window="24h",          # 唯一支持的窗口
)

# 3. 轮询直到完成（几分钟到 24 小时），然后下载输出文件。
batch = client.batches.retrieve(batch.id)     # -> status: validating/in_progress/completed
if batch.status == "completed":
    results = client.files.content(batch.output_file_id).text  # .jsonl，每行一个结果
```

下面那张餐巾纸算术单元格里的经济账，就是值得折腾的全部理由：*所有东西*约 5 折，还能和便宜模型叠加，适用于回填和批量 eval。代价是延迟和一点管道工作（jsonl 进、轮询、jsonl 出），在没人在等的时候，这微不足道。

## 你该能徒手算出来的餐巾纸算术

你现在已经见过三种成本杠杆了。在算术之前，先把它们并排摆出来：各自针对什么、大致折扣、以及何时该用：

| 杠杆 | 针对什么 | 大致效果 | 取舍 | 什么时候用它 |
|---|---|---|---|---|
| **prompt 缓存** | 重复的*前缀* | 缓存 token 约 0.1 倍（写入时一次性约 1.25 倍） | TTL 短；要求精确前缀匹配的纪律 | 一大段 system prompt / 文档在多次调用间重复 |
| **Batch API** | 请求里的*所有东西* | 全部 token 约 0.5 倍 | 异步，几分钟到 24 小时延迟 | 没人在等：回填、批量 eval、每夜任务 |
| **选模型** | *费率*本身 | 档次之间常常差 3-5 倍 | 小模型可能通不过你的 eval | 便宜档和贵档通过同一个 eval |

它们**按乘法叠加**：小模型 × batch × 缓存前缀，通常比一个不带缓存的实时大模型便宜 20-50 倍。而每一种都归结为同一个公式：`tokens × rate`，带三个乘数（缓存写入约 1.25 倍输入、缓存读取约 0.1 倍输入、batch 对所有东西 0.5 倍）。一个 AI 工程师如果不能在餐巾纸上勾出这个，就会被账单吓一跳；能勾出来的，会在设计评审里一眼发现那 10 倍的节省。

下面的费率是示意性的每 token 价格（Groq 免费档是 0 美元，而缓存/batch 在那里并未提供）；重点是算术的*形状*，它在每一个按 token 计费的厂商那里都完全一样。先看辅助函数，再看两个算好的例子。

In [ ]:
PRICES = {
    # 每百万 token 的美元价：(输入, 输出)。示意性费率——重要的是这个形状。
    'small':  (1.00, 5.00),    # 便宜/批量档
    'mid':    (3.00, 15.00),   # 迭代默认档
    'large':  (5.00, 25.00),   # 重推理档
}


def estimate_cost(model, input_tokens=0, output_tokens=0,
                  cache_read=0, cache_write=0, batch=False):
    inp, out = PRICES[model]
    usd = (input_tokens * inp
           + output_tokens * out
           + cache_read * inp * 0.10
           + cache_write * inp * 1.25) / 1_000_000
    return usd * (0.5 if batch else 1.0)


# 例 1：一个客服机器人。每天 1 万次请求，8k token 的 runbook 前缀，200 token 的
# 问题，300 token 的回答。对比无缓存与缓存前缀（都在 'mid' 档上）。
n = 10_000
no_cache = n * estimate_cost('mid', input_tokens=8_200, output_tokens=300)
cached = (estimate_cost('mid', cache_write=8_000, input_tokens=200, output_tokens=300)
          + (n - 1) * estimate_cost('mid', cache_read=8_000, input_tokens=200, output_tokens=300))
print(f'support bot / day  no cache: ${no_cache:,.2f}   cached: ${cached:,.2f}   '
      f'saving: {1 - cached / no_cache:.0%}')

# 例 2：给 100 万张工单做回填分类（每张 500 进 / 10 出），用 small 档，batch 对比实时。
rt = 1_000_000 * estimate_cost('small', input_tokens=500, output_tokens=10)
bt = 1_000_000 * estimate_cost('small', input_tokens=500, output_tokens=10, batch=True)
print(f'1M-ticket backfill  realtime: ${rt:,.2f}   batch: ${bt:,.2f}')

# 还有例 2 的错配模型版本，因为总有人会提这个方案：
large_rt = 1_000_000 * estimate_cost('large', input_tokens=500, output_tokens=10)
print(f'same backfill on realtime large tier: ${large_rt:,.2f}  <- this is the design-review catch')

跑一下，注意省钱的形状：缓存攻击的是*重复的前缀*（共享上下文很大时占主导），batch 攻击的是*所有东西*（对接量任务占主导），而选模型的影响把这两者都盖过去了。三者相乘：小模型 + batch + 缓存前缀，通常比一个不带缓存的实时大模型便宜 20-50 倍，前提是这件工作里便宜版本能通过同一个 eval。

## 练习

1. 把 `tiktoken` 估算和真实计数做对比：用 `count(...)` 编码若干个长度不同的 prompt，每个都以 `max_tokens=1` 发送，然后把估算值和 `usage.prompt_tokens` 比较。对你正在调用的模型，tiktoken 有多大偏差？
2. 写一个 `fit_history(history, budget_tokens)`，从最旧的回合开始丢弃（始终以 user/assistant 成对丢弃），直到 `count(history)` 装进预算。在一段编造的 20 回合对话上测试它。
3. 扩展 `estimate_cost`，让它接受 `calls_per_day` 和 `cache_hit_rate`，为三个档次分别产出带缓存和不带缓存的月度预测表。手工核对其中一行。
4. 度量 Groq 的真实延迟与 token 数的关系：分别在 `max_tokens` 为 50、200、800 时给请求计时，确认总延迟随输出 token 增长。这就是「预留输出余量」这一课的具体体现。